<a href="https://colab.research.google.com/github/Decoding-Data-Science/airesidency/blob/main/crewai_multiagent_level_1-18thjul.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# workshop: Create Multi Agents to Research and Write an Article

In this workshop, you will be introduced to the foundational concepts of multi-agent systems and get an overview of the crewAI framework.

https://huggingface.co/spaces/decodingdatascience/fromprompttopost

In [1]:
#install packages
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29

In [2]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import from the crewAI libray.

In [3]:
from crewai import Agent, Task, Crew

- As a LLM for your agents, you'll be using OpenAI's `gpt-3.5-turbo`.



In [5]:
#setting the model
import os

#os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'
os.environ["OPENAI_MODEL_NAME"] = 'gpt-4.1-nano-2025-04-14'

## Creating Agents

- Define your Agents, and provide them a `role`, `goal` and `backstory`.
- It has been seen that LLMs perform better when they are role playing.

### Agent: Planner

**Note**: The benefit of using _multiple strings_ :
```Python
varname = "line 1 of text"
          "line 2 of text"
```

versus the _triple quote docstring_:
```Python
varname = """line 1 of text
             line 2 of text
          """
```
is that it can avoid adding those whitespaces and newline characters, making it better formatted to be passed to the LLM.

In [6]:
# Retrieve API key from Colab secrets and set it in environment variables
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('openai')

In [7]:
from crewai_tools import (
    DirectoryReadTool,
    FileReadTool,
    SerperDevTool,
    WebsiteSearchTool
)

In [9]:
# Set up API keys - enter from https://serper.dev/billing
os.environ["SERPER_API_KEY"] = "218873db9b73d345d3083e2f1f29cb74b14d4ddb" # serper.dev API key

# Instantiate tools
docs_tool = DirectoryReadTool(directory='./data')
file_tool = FileReadTool()
search_tool = SerperDevTool()
web_rag_tool = WebsiteSearchTool()

In [11]:
planner = Agent(
    role="Content Planner",
    goal="Plan engaging and factually accurate content on {topic}",
    backstory="You're working on planning a blog article "
              "about the topic: {topic}."
              "You collect information that helps the "
              "audience learn something "
              "and make informed decisions. "
              "Your work is the basis for "
              "the Content Writer to write an article on this topic."    ,
    tools=[search_tool, web_rag_tool,docs_tool,file_tool],
    allow_delegation=False,
	verbose=True
)

### Agent: Writer

In [12]:
writer = Agent(
    role="Content Writer",
    goal="Write insightful and factually accurate "
         "opinion piece about the topic in 1500 words: {topic}",
    backstory="You're working on a writing "
              "a new opinion piece about the topic: {topic}. "
              "You base your writing on the work of "
              "the Content Planner, who provides an outline "
              "and relevant context about the topic. "
              "You follow the main objectives and "
              "direction of the outline, "
              "as provide by the Content Planner. "
              "You also provide objective and impartial insights "
              "and back them up with information "
              "provide by the Content Planner. "
              "You acknowledge in your opinion piece "
              "when your statements are opinions "
              "as opposed to objective statements.",
    allow_delegation=False,
    verbose=True
)

### Agent: Editor

In [13]:
editor = Agent(
    role="Editor",
    goal="Edit a given blog post to align with "
         "the writing style of the organization. ",
    backstory="You are an editor who receives a blog post "
              "from the Content Writer. "
              "Your goal is to review the blog post "
              "to ensure that it follows journalistic best practices,"
              "provides balanced viewpoints "
              "when providing opinions or assertions, "
              "and also avoids major controversial topics "
              "or opinions when possible.",
    allow_delegation=False,
    verbose=True
)

## Creating Tasks

- Define your Tasks, and provide them a `description`, `expected_output` and `agent`.

### Task: Plan

In [22]:
plan = Task(
    description=(
        "1. Prioritize the latest trends, key players, "
            "and noteworthy news on {topic}.\n"
        "2. Identify the target audience, considering "
            "their interests and pain points.\n"
        "3. Develop a detailed content outline including "
            "an introduction, key points, and a call to action.\n"
        "4. Include SEO keywords and relevant data or sources."
    ),
    expected_output="A comprehensive content plan document "
        "with an outline, audience analysis, "
        "SEO keywords, and resources.",
    agent=planner,
)

### Task: Write

In [16]:
from datetime import date
current_date = date.today().strftime("%B %d, %Y")

In [17]:
current_date

'July 18, 2026'

In [18]:
from datetime import date

current_date = date.today().strftime("%B %d, %Y")

write = Task(
    description=(
        f"use Today's Date: {current_date}\n\n"
        "1. Use the content plan to craft a compelling "
            "blog post on {topic}.\n"
        "2. Incorporate SEO keywords naturally.\n"
		"3. Sections/Subtitles are properly named "
            "in an engaging manner.\n"
        "4. Ensure the post is structured with an "
            "engaging introduction, insightful body, "
            "and a summarizing conclusion.\n"
        "5. Proofread for grammatical errors and "
            "alignment with the brand's voice.\n"
    ),
    expected_output="A well-written blog post "
        "in markdown format, ready for publication, "
        "each section should have 2 or 3 paragraphs.1500 words",
    agent=writer,
)

### Task: Edit

In [20]:
edit = Task(
    description=("Proofread the given blog post for "
                 "grammatical errors and "
                 "alignment with the brand's voice."),
    expected_output="A well-written blog post in markdown format, "
                    "ready for publication, "
                    "each section should have 2 or 3 paragraphs.",
    agent=editor
)

## Creating the Crew

- Create your crew of Agents
- Pass the tasks to be performed by those agents.
    - **Note**: *For this simple example*, the tasks will be performed sequentially (i.e they are dependent on each other), so the _order_ of the task in the list _matters_.
- `verbose=2` allows you to see all the logs of the execution.

In [23]:
crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    planning=True,
    verbose=4
)

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

## Running the Crew
## Try it Yourself

- Pass in a topic of your choice and see what the agents come up with!

In [24]:
topic = "who will win Fifa 2026"
result = crew.kickoff(inputs={"topic": topic})

 [DEBUG]: == Working Agent: Content Planner
 [INFO]: == Starting Task: 1. Prioritize the latest trends, key players, and noteworthy news on who will win Fifa 2026.
2. Identify the target audience, considering their interests and pain points.
3. Develop a detailed content outline including an introduction, key points, and a call to action.
4. Include SEO keywords and relevant data or sources.


> Entering new CrewAgentExecutor chain...


Thought: To develop a comprehensive content plan about "Who Will Win FIFA 2026," I need to gather current trends, expert opinions, and key player analysis related to the upcoming tournament. Since I do not have direct access to recent articles or predictions, I should search the internet for the latest insights and forecasts. Additionally, understanding the target audience will help tailor the content effectively. I will start by searching the internet for recent discussions, expert predictions, and key factors influencing FIFA 2026 outcomes.

Action: Search the internet

Action Input: {"search_query": "FIFA 2026 winner predictions, latest trends, expert opinions, key players"}

 


Search results: Title: Who Will Win the 2026 FIFA World Cup? The Opta Supercomputer ...
Link: https://theanalyst.com/articles/who-will-win-2026-fifa-world-cup-predictions-opta-supercomputer
Snippet: Spain are the most likely winners of the 2026 World Cup, winning the tournament in 16.1% of the 25,000 pre-tournament simulations by the Opta ...
---
Title: World Cup predictions: Picking the winner in every game of the entire ...
Link: https://www.espn.com/soccer/story/_/id/48962628/world-cup-predictions-picking-winner-every-game-entire-tournament
Snippet: Brazil have the fourth-best odds of winning the World Cup -- after the three consensus favorites: Spain, France and England.
---
Title: Supercomputer Predicts 2026 World Cup Winner
Link: https://www.si.com/soccer/supercomputer-predicts-2026-world-cup-winner
Snippet: Lamine Yamal, Ferran Torres Spain could seal a famous victory this summer. Spain is given a 16.02% chance of going all the way this summer.
---
Title: World Cup Bold Pred

Thought: I need to gather recent and relevant insights about potential winners of the FIFA 2026 World Cup, focusing on expert predictions, simulations, and key team analyses. The search results indicate that Spain, England, France, and Brazil are among the top contenders, with Spain often highlighted as a favorite in recent simulations. Additionally, understanding the factors influencing these predictions, such as team form, star players, and tournament location, will be important. I should also consider the target audience—football fans, bettors, sports analysts—interested in informed predictions and analysis. Next, I will analyze the most credible sources and synthesize this information into a structured content plan.

Thought: I will now read the most relevant sources to extract detailed insights about the predictions, key teams, and factors influencing the outcome of FIFA 2026.

Action: Read a file's content

Action Input: {"file_path": "./data/latest_fifa_predictions.html"} 

I en

Thought: To develop a comprehensive content plan for "Who Will Win FIFA 2026," I need to synthesize recent expert predictions, simulation results, and key team analyses. The search results indicate that Spain is frequently highlighted as a top contender, often predicted to win based on simulations like those from Opta and other predictive models. Other strong contenders include England, France, and Brazil, with some sources giving them high probabilities. Additionally, understanding the factors influencing these predictions—such as team form, star players, tournament location, and historical performance—will help craft a nuanced article.

The target audience includes football fans, sports bettors, analysts, and general sports enthusiasts interested in informed predictions and tournament insights. They seek credible forecasts, key team strengths, and potential dark horses.

I will organize content around recent predictive insights, highlight the leading teams and their chances, discuss 

Final Answer:

# Who Will Win FIFA 2026? An In-Depth Analysis of the Contenders and Predictions

## Introduction

Since its inception in 1930, the FIFA World Cup has grown to become the most prestigious and widely watched sporting event globally. It unites nations, sparks passionate debates, and showcases the best talents in football. Each edition brings new surprises, rising stars, and legendary moments that etch themselves into football history. As the anticipation for FIFA 2026 builds—set to be hosted across the United States, Canada, and Mexico—the football world is abuzz with predictions and analyses about who will lift the trophy.

Predicting the outcome of a tournament held three years into the future involves a mix of current form, historical trends, and sophisticated models. Fans, bettors, and experts alike are eager to speculate on the likely winner, with many turning to advanced simulations and statistical forecasts to inform their opinions. This article aims to dissect the 

In [25]:
from IPython.display import Markdown
Markdown(result)

# Who Will Win FIFA 2026? An In-Depth Analysis of the Contenders and Predictions

## Introduction

Since its debut in 1930, the FIFA World Cup has established itself as the pinnacle of international football, captivating fans around the globe and uniting nations through the beautiful game. Each edition introduces fresh narratives—rising stars, unforgettable moments, and unexpected results—adding to the tournament’s legendary status. As anticipation for FIFA 2026 builds, with the event set to be hosted across the United States, Canada, and Mexico, excitement and speculation are reaching new heights. Fans and analysts alike are eager to explore which teams are best positioned to claim the coveted trophy.

Predicting the winner of a tournament that is still three years away involves balancing current team form, historical trends, and cutting-edge analytical models. While no prediction can be foolproof, recent advancements in data analysis and simulation provide valuable insights. This article aims to examine the most credible forecasts, analyze key contenders, and consider the factors that could sway the outcome of FIFA 2026, all while maintaining a balanced and objective perspective.

## Current Trends and Predictions

Recent predictive models, including sophisticated simulations from sources like *The Analyst*, suggest that Spain may enter the tournament as a strong contender. These supercomputer-based forecasts estimate Spain’s chances of winning at around 16%, based on detailed analyses of team performances, player statistics, historical data, and upcoming fixtures. Such models are designed to incorporate a wide array of variables, aiming to provide a probabilistic outlook on the tournament’s outcome.

However, traditional football powerhouses—such as England, France, Brazil, and Germany—continue to feature prominently in predictions. These teams benefit from recent international performances, squad depth, tactical versatility, and experienced coaching staff. While simulation models are increasingly accurate, they cannot account for every unpredictable factor—such as injuries, tactical adjustments, or emergent talents—that could influence results. As the tournament approaches, real-world elements like team chemistry, player fitness, and on-field performance will remain crucial determinants of success.

## Key Teams and Their Chances

### Spain: The Leading Contender

Spain’s recent performances reaffirm its status among the top favorites for FIFA 2026. Known for their possession-based, tactical style of play, the Spanish team boasts technically skilled players and a robust youth development system that continuously produces emerging talent. Their international record includes a World Cup victory in 2010 and consistent deep runs in major tournaments. If Spain maintains tactical discipline and adapts to the evolving global football landscape, they could be strong contenders for the title.

The current squad features a blend of experienced internationals and promising young players, offering both stability and dynamism. Their strategic flexibility and innovative coaching will be vital as they navigate a highly competitive field. While their historical success provides confidence, sustained performance and squad cohesion in the lead-up to 2026 will be key factors in realizing their championship ambitions.

### England: Consistent Top Contender

England remains firmly in the conversation as a favorite heading into FIFA 2026. With a wealth of talent competing in top European clubs, the Three Lions possess a potent mix of skill, tactical discipline, and international experience. Their recent performances—reaching the semi-finals of the 2018 World Cup and a strong showing at Euro 2020—highlight their resilience and capacity to contend at the highest level.

England’s balanced approach—combining attacking flair with defensive solidity—coupled with the emergence of young talents, makes them a formidable opponent. However, maintaining consistency and team cohesion will be critical, especially given the tournament’s demanding schedule and the pressure to perform on the global stage. Their ability to manage high expectations and adapt tactically will influence their chances of lifting the trophy.

### France: The Defending Champions

As the reigning World Cup champions from 2018, France enters FIFA 2026 with significant momentum. Led by star players like Kylian Mbappé, and supported by a deep squad of internationals, France’s talent pool is arguably among the strongest in the tournament. Their blend of youthful exuberance and experienced leadership provides a competitive edge, making them perennial favorites.

Tactical versatility and a proven winning mentality from their previous success further bolster their prospects. Nonetheless, challenges such as potential injuries, squad rotations, or internal dynamics could influence their performance. Staying healthy and maintaining team harmony will be essential for France to defend their title and aim for back-to-back championships.

### Brazil: The Historic Powerhouse

Brazil’s footballing legacy is unmatched, with five World Cup titles and a reputation for producing creative, technically gifted players. Their vibrant, attacking style of play and passionate fanbase consistently make them contenders. Recent efforts to integrate emerging talents alongside seasoned internationals suggest Brazil is preparing for another deep run in 2026.

While their squad may lack the experience of past dominant generations, their raw talent and tactical adaptability keep them in the mix. The unpredictability of their attacking flair and the ability to generate moments of individual brilliance make Brazil a perennial threat to any opponent. If their young players develop cohesively and they execute their game plan effectively, they remain strong candidates for the title.

### Other Notable Contenders

Beyond the top favorites, teams like Germany, Argentina, and potential dark horses such as Belgium or the Netherlands are worth monitoring. Germany’s disciplined, tactical approach and strong youth system make them consistent threats, capable of surprising at major tournaments. Argentina, with its blend of seasoned veterans and emerging stars, always holds potential for a deep run.

Historically, underdog teams leveraging strategic planning, team cohesion, and fortunate draws have delivered surprises. Teams like Belgium or the Netherlands, boasting talented squads and tactical ingenuity, could emerge unexpectedly if they peak at the right moment. Such unpredictability is part of what makes the World Cup so exciting.

## Influencing Factors

### Hosting Countries and Tournament Location

FIFA 2026’s expanded format—featuring 48 teams across three host nations—introduces additional variables into the competition. The United States’ extensive infrastructure and organizational capacity promise a smooth tournament experience. Meanwhile, Mexico’s passionate football culture and favorable climate can provide a significant advantage for CONCACAF teams.

Travel logistics, climate adaptation, and crowd support are factors that could influence team performance. For example, teams accustomed to playing in different time zones or weather conditions may find certain venues more favorable. While the tournament’s global nature aims to level the playing field, these logistical considerations could still impact outcomes.

### Team Form and Injuries

As the tournament approaches, the current form of teams will be a crucial determinant of success. Injuries to key players—such as Neymar, Harry Kane, or other star talents—could significantly alter a team’s chances. Historically, injuries to pivotal players have shifted the balance of power in major tournaments.

Monitoring squad health and readiness will be essential for accurate predictions. Coaches’ tactical adjustments, squad depth, and the ability to adapt under pressure will also play vital roles in determining which teams can perform at their best when it matters most.

### Star Players and Emerging Talents

The influence of standout individuals remains a defining aspect of World Cup tournaments. Players like Kylian Mbappé, Harry Kane, and emerging talents such as Jude Bellingham or Gavi can be game-changers on any given day. Their ability to produce moments of brilliance can tilt matches and inspire their teams.

Coaches’ capacity to harness and motivate these star players will be pivotal. The tournament often features young talents stepping into the spotlight, adding unpredictability and excitement. Their performances could be decisive in tight matches and ultimately influence the tournament’s outcome.

### Tactical Trends and Coaching Strategies

Modern football is increasingly characterized by tactical flexibility and innovative coaching. Teams that adapt quickly to evolving styles, utilize data analytics, and implement effective game plans will have a competitive edge. Experienced coaches like Didier Deschamps, Gareth Southgate, and emerging tactical minds will play critical roles in guiding their teams through the challenges of the tournament.

Strategic decisions—such as formation changes, substitutions, and game management—can be the difference between victory and defeat. The ability to exploit opponent weaknesses and respond to in-game developments will be central to a team’s success in the knockout stages.

## Expert Opinions and Media Predictions

Leading sports analysts from outlets such as *BBC*, *ESPN*, and betting experts offer varied perspectives on FIFA 2026 favorites. While there is consensus that teams like France, England, and Spain are among the top contenders, the overall predictions highlight the tournament’s inherent unpredictability. The expanded format and the presence of talented underdogs mean surprises are highly likely.

Betting odds reflect these sentiments, with France and England generally favored, followed closely by Spain and Brazil. Experts advise caution, emphasizing that simulation models provide valuable insights but are not definitive. Real-world factors—team cohesion, tactical adaptability, and unforeseen events—often shape the final result more than predictions alone.

## Future Outlook and Dark Horses

Although established teams currently dominate the predictions, history has demonstrated that the World Cup often produces unexpected champions. Emerging teams that qualify through regional tournaments can develop into dark horses if they display strong tactical organization and team cohesion. Past tournaments have shown that experience, luck, and strategic execution are often decisive.

Teams with a promising blend of seasoned veterans and talented youth—such as Belgium or the Netherlands—could surprise favorites if they perform at their peak. Additionally, unforeseen injuries, tactical innovations, or fortunate draws might elevate lesser-known nations into the spotlight, adding to the tournament’s excitement and unpredictability.

## Conclusion & Call to Action

Based on current data, simulation forecasts, and expert insights, **Spain** appears to be the most probable winner of FIFA 2026. However, strong competition from **France**, **Brazil**, and **England** ensures a highly competitive and unpredictable tournament. The expanded format, diverse participating nations, and the inherent unpredictability of football guarantee that the eventual champion may defy early expectations.

Fans and bettors should stay informed by monitoring team developments, injury reports, and tactical trends. Combining statistical models with on-the-ground insights will provide a balanced approach to predictions. The 2026 World Cup promises a spectacular showcase of talent, resilience, and strategic mastery—where any team could emerge victorious.

Remember, football’s beauty lies in its unpredictability. Engage with fellow fans, share your predictions, and enjoy every moment of this grand sporting celebration in North America. Whether you’re placing bets or simply watching the drama unfold, the journey to the 2026 FIFA World Cup will be filled with excitement and unforgettable moments.

---

**SEO Keywords integrated naturally:**  
FIFA 2026 winner prediction, 2026 World Cup favorites, Who will win FIFA 2026, FIFA World Cup 2026 predictions, Best teams for FIFA 2026, 2026 World Cup odds, Top contenders FIFA 2026, FIFA 2026 betting tips

---

## Resources & References

- [The Analyst: Who Will Win the 2026 FIFA World Cup?](https://theanalyst.com/articles/who-will-win-2026-fifa-world-cup-predictions-opta-supercomputer)  
- [SI: Supercomputer Predicts 2026 Winner](https://www.si.com/soccer/supercomputer-predicts-2026-world-cup-winner)  
- [ESPN: World Cup Predictions](https://www.espn.com/soccer/story/_/id/48962628/world-cup-predictions-picking-winner-every-game-entire-tournament)  
- [BBC: Expert Predictions](https://www.bbc.com/sport/football/articles/cg7485gyxz9o)  
- [Science Focus: Scientific Models & Predictions](https://www.sciencefocus.com/news/win-world-cup-prediction)

# Deploy this crew in gradio

In [ ]:
pip install gradio

In [ ]:
import gradio as gr
from crewai import Agent, Task, Crew

# Define Agents
researcher = Agent(
    role="Senior Research Specialist",
    goal="Uncover intricate insights into the given topic",
    backstory="A seasoned expert in extracting valuable information and providing detailed analysis on any given subject.",
    allow_delegation=False
)

writer = Agent(
    role="Content Writer",
    goal="Craft engaging and informative articles",
    backstory="An experienced content writer who can turn complex research into easy-to-understand and engaging articles.",
    allow_delegation=False
)

editor = Agent(
    role="Content Editor",
    goal="Ensure articles are polished, coherent, and engaging",
    backstory="An expert editor with a sharp eye for clarity, grammar, and flow, capable of turning drafts into publication-ready articles.",
    allow_delegation=False
)

# Function to setup tasks and run CrewAI
def run_crewai_article_writer(topic):
    # Define tasks
    research_task = Task(
        description=f"Conduct in-depth research on the topic: {topic} and provide a structured summary.",
        expected_output="A structured and detailed research summary covering important aspects, recent developments, and insights related to the topic.",
        agent=researcher
    )

    writing_task = Task(
        description=f"Using the research, write a full-length article on the topic: {topic} in 1500 words give various sub section",
        expected_output="An engaging, detailed, informative, and coherent article suitable for online publication.",
        agent=writer
    )

    editing_task = Task(
        description=f"Review and edit the article on {topic} for clarity, grammar, flow, and engagement.",
        expected_output="A polished, clear, and professionally edited final version of the article.",
        agent=editor
    )

    # Setup Crew
    crew = Crew(
        agents=[researcher, writer, editor],
        tasks=[research_task, writing_task, editing_task],
        verbose=True  # Set to False if you want less console output
    )

    # Execute
    result = crew.kickoff()
    return result

# Gradio Interface
def generate_article(topic):
    return run_crewai_article_writer(topic)

iface = gr.Interface(
    fn=generate_article,
    inputs=gr.Textbox(lines=2, placeholder="Enter a topic for the article..."),
    outputs=gr.Textbox(lines=20, label="Generated Article"),
    title="AI Agent: Prompt to Post",
    description="Enter a topic and let CrewAI agents research, write, and edit a complete article!"
)

if __name__ == "__main__":
    iface.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f32acd2d3790bf834e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
pip install langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.3/378.3 kB 21.3 MB/s eta 0:00:00
  Attempting uninstall: langsmith
    Found existing installation: langsmith 0.1.147
    Uninstalling langsmith-0.1.147:
      Successfully uninstalled langsmith-0.1.147
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.1.53
    Uninstalling langchain-core-0.1.53:
      Successfully uninstalled langchain-core-0.1.53
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-cohere 0.1.5 requires langchain-core<0.3,>=0.1.42, but you have langchain-core 1.3.0 which is incompatible.
langchain-community 0.0.29 requires langchain-core<0.2.0,>=0.1.33, but you have langchain-core 1.3.0 which is incompatible.
langchain-community 0.0.29 requires langsmith<0.2.0,>=0.1.0, but